In [1]:
"""
Script Name: model_demanda_atendida_renovable.py
Authores: Walter Arredondo, Carlos Daza, Andrés Coral
Fecha: 2026-10-03
Modelo de Predicción
Descripción:
Se define la variable objetivo Demanda Atendida mediante Fuentes Renovables y se obtiene el mejor
modelo de predicción con base a las metricas de evaluación MAE-MSE-RMSE-R².
Entradas: - CSV final load: data/processed/
Salida: - Modelo de Predicción: data/model/
Notas:
"""

'\nScript Name: model_demanda_atendida_renovable.py\nAuthores: Walter Arredondo, Carlos Daza, Andrés Coral\nFecha: 2026-10-03\nModelo de Predicción\nDescripción:\nSe define la variable objetivo Demanda Atendida mediante Fuentes Renovables y se obtiene el mejor\nmodelo de predicción con base a las metricas de evaluación MAE-MSE-RMSE-R².\nEntradas: - CSV final load: data/processed/\nSalida: - Modelo de Predicción: data/model/\nNotas:\n'

#Recuperar información load

In [2]:
import pandas as pd

In [3]:
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
ruta = "/content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/processed/energia.csv"
df_final = pd.read_csv(ruta, sep = ';')

# Modelo para la Predicción de Demanda Energetica

Este Notebook realiza


La demanda Energetica de un Pais representa la cantidad total de energía que los usuarios (hogares, industrias y comercios) requieren en un momento específico para operar sus equipos y satisfacer sus necesidades. La capacidad de cubrir dicho consumo mediante energía renovable depende de la disponibilidad instantánea o anual de la generación de las fuentes las cuales son:


*   Fuente Eólica
*   Fuente Solar
*   Fuente hidroeléctrica
*   Fuente geotérmica
*   Fuente biomasa

Un país no puede consumir más energía renovable de la que genera, ni tampoco puede "atender" masa energía renovable que la que los usuarios demandan.

Se define la variable Demanda Atendida mediante Fuentes Renovables



*   Demanda_Atendida_Renovable= min(Demanda_Electrtica_Total,  Generación_Renovable_Total)

Se considera los dos escenarios fundamentales del balance energético



*   Déficit Renovable (Generación < Demanda)
*   Elemento de lista (Generación ≥ Demanda)







In [5]:
#semilla
import random
random.seed(123)

In [6]:
import numpy as np

In [7]:
# Calculo Demanda Atendida Renovable

# Agrupar columnas consumo
cols_consumo = ['consumo_agro_pesca_y_mineria',
                'consumo_comercial_servicios_publico',
                'consumo_construccion_y_otros',
                'consumo_industrial',
                'consumo_residencial',
                'consumo_transporte']

# Agrupar columnas generacion renovable
cols_renovables = [
    'generacion_eolica', 'generacion_geotermia',
    'generacion_hidro', 'generacion_solar',
    'generacion_termica_renovable'
]

#calculos principales
df_final['demanda_electrica_total'] = df_final[cols_consumo].sum(axis=1)
df_final['generacion_renovable_total'] = df_final[cols_renovables].sum(axis=1)
# calculo variable objetivo
df_final['demanda_atendida_renovable'] = np.minimum(df_final['demanda_electrica_total'], df_final['generacion_renovable_total'])
df = df_final.copy()

Durante la Fase de Análisis Exploratorio de Datos (EDA) se identificaron diferencias en las escalas de las variables continuas, lo que sugería la necesidad de un escalado o normalización. Pero ya que se esta trabajando con algoritmos que son invariantes a las variaciones de escalas y/o otliners se decidio preservar con el fin de mantener la interpretabilidad directa de los diferntes indicadores temporales de cada Pais.

## Lazypredict
Con el fin de entrenar y comparar decenas de modelos de aprendizaje automático se utiliza la libreria de python Lazypredict

In [8]:
!pip install lazypredict

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 1.5 MB/s eta 0:00:00


In [9]:
# librerias necesarias
from lazypredict.Supervised import LazyRegressor
from sklearn.model_selection import train_test_split

#Preprocesamiento para evitar fuga de datos
Puesto que la variable a predecir o target es demanda_atendida_renovable, es indispensable excluir ciertas columnas con el fin de evitar fuga de datos y redudancia matematica para obtener una buena predicción

In [10]:
# columnas de consumo
cols_consumo = [c for c in df.columns if c.startswith('consumo_')]
# columnas de totales
cols_excluir = ['demanda_electrica_total', 'generacion_renovable_total'] + cols_consumo

In [11]:
# dataset para modelo predictivo
#eliminar columnas y posibles faltantes o nulos (aunque eso ya se trato en la fase de trasformación)
df_ml = df.drop(columns=cols_excluir).fillna(0)
#variables categoricas en numéricas One hot Encoding
df_ml = pd.get_dummies(df_ml, columns=['pais'], drop_first=True)

In [12]:

# Se define X y Y
X = df_ml.drop(columns=['demanda_atendida_renovable'])
y = df_ml['demanda_atendida_renovable']
# Se separa el dataset en entrenamiento y test (Entrenamiento <= 2017 | Evaluación > 2017)
train_mask = df_ml['anio'] <= 2017
test_mask = df_ml['anio'] > 2017
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Entrenar y evaluar más de 30 modelos con LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

# Mostrar la tabla comparativa ordenada por R-Squared
print(models)

                               Adjusted R-Squared     R-Squared          RMSE  \
Model                                                                           
LassoLarsIC                         -6.516360e-01  5.191440e-01  1.556859e+00   
KNeighborsRegressor                 -7.204977e-01  4.990956e-01  1.588983e+00   
OrthogonalMatchingPursuit           -7.271211e-01  4.971673e-01  1.592038e+00   
LarsCV                              -7.579260e-01  4.881987e-01  1.606173e+00   
RandomForestRegressor               -9.758965e-01  4.247390e-01  1.702841e+00   
TweedieRegressor                    -1.012030e+00  4.142190e-01  1.718341e+00   
BaggingRegressor                    -1.132558e+00  3.791287e-01  1.769060e+00   
AdaBoostRegressor                   -1.316271e+00  3.256425e-01  1.843685e+00   
LassoCV                             -1.490662e+00  2.748706e-01  1.911831e+00   
LassoLarsCV                         -1.492322e+00  2.743871e-01  1.912468e+00   
ExtraTreeRegressor          

Con base al coefciente de variazión o $R^2$ que mide la varibilidad de los datos frente a la explicación del modelo se determina que los 3 mejores modelos son:


*   LassoLarsIC
*   KNeighborsRegressor
*   OrthogonalMatchingPursuit



##  Featuring Engineering

Ya que se tienen  secuencias de datos u observaciones de una variable tomados en momentos específicos, ordenados de manera cronológica y separados por intervalos (Años) se puede aplicar Ingenieria de caracteristicas para series Temporales con el fin de enseñarle al modleo el historial del pasado y las tendencias de los datos

In [13]:
#dataset sin modificar
df2_ml = df.copy()
# columnas de consumo
cols_consumo = [c for c in df.columns if c.startswith('consumo_')]
# columnas de totales
cols_excluir = ['demanda_electrica_total', 'generacion_renovable_total'] + cols_consumo
#eliminar columnas y posibles faltantes o nulos (aunque eso ya se trato en la fase de trasformación)
df2_ml = df2_ml.drop(columns=cols_excluir).fillna(0)

In [14]:

# Variables de Retardo (Lags) valores del pasado
#demanda atendida hace un año t-1
df2_ml['lag_1'] = df2_ml.groupby('pais')['demanda_atendida_renovable'].shift(1)
#demanda atendida hace dos años t-2
df2_ml['lag_2'] = df2_ml.groupby('pais')['demanda_atendida_renovable'].shift(2)
#demanda atendida hace tres años t-3
df2_ml['lag_3'] = df2_ml.groupby('pais')['demanda_atendida_renovable'].shift(3)

# Crecimiento/Variación respecto al año anterior - tendencia de la demanda
df2_ml['diff_1'] = df2_ml['lag_1'] - df2_ml['lag_2']

# Promedio y desviación estándar móvil a 3 años ( estadisticas moviles)
# Extraer el nivel general/sotenido de demanda
df2_ml['roll_mean_3'] = df2_ml.groupby('pais')['demanda_atendida_renovable'].shift(1).rolling(3).mean()
# Medir que tan variable/estable ha sido la demanda en los ultimos 3 años (identificar periodos de tiempo inestables)
df2_ml['roll_std_3'] = df2_ml.groupby('pais')['demanda_atendida_renovable'].shift(1).rolling(3).std().fillna(0)

# Limpiar nulos
df2_ml = df2_ml.dropna().reset_index(drop=True)
# One-Hot Encoding
df2_ml = pd.get_dummies(df2_ml, columns=['pais'], drop_first=True)

In [15]:
# Se define X y Y 2 entrenamiento
X = df2_ml.drop(columns=['demanda_atendida_renovable'])
y = df2_ml['demanda_atendida_renovable']
# Se separa el dataset en entrenamiento y test (Entrenamiento <= 2017 | Evaluación > 2017)
train_mask = df2_ml['anio'] <= 2017
test_mask = df2_ml['anio'] > 2017
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Entrenar y evaluar más de 30 modelos con LazyRegressor
reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
models, predictions = reg.fit(X_train, X_test, y_train, y_test)

# Mostrar la tabla comparativa ordenada por R-Squared
print(models)

                               Adjusted R-Squared     R-Squared          RMSE  \
Model                                                                           
ExtraTreesRegressor                  7.186265e-01  9.394513e-01  5.524515e-01   
GradientBoostingRegressor            6.288330e-01  9.201286e-01  6.345083e-01   
XGBRegressor                         4.293989e-01  8.772124e-01  7.867177e-01   
BaggingRegressor                     4.179368e-01  8.747459e-01  7.945801e-01   
LGBMRegressor                        4.040872e-01  8.717656e-01  8.039776e-01   
HistGradientBoostingRegressor        4.027484e-01  8.714775e-01  8.048803e-01   
RandomForestRegressor                3.688657e-01  8.641863e-01  8.273961e-01   
OrthogonalMatchingPursuit            3.468751e-01  8.594541e-01  8.416872e-01   
OrthogonalMatchingPursuitCV          3.468751e-01  8.594541e-01  8.416872e-01   
AdaBoostRegressor                    3.251773e-01  8.547850e-01  8.555540e-01   
HuberRegressor              

Despues de aplicar Featuring Engineering a series temporales se obtuvo los resultados  coefciente de variazión o  R2. Se determino que los 3 mejores modelos son:



*   ExtraTreesRegressor
*   GradientBoostingRegressor
*   BaggingRegressor



Se genera el modelo ganador

In [16]:

from sklearn.ensemble import ExtraTreesRegressor

In [17]:
modelo_f = ExtraTreesRegressor(random_state=42)
modelo_f.fit(X_train, y_train)

ExtraTreesRegressor(random_state=42)

#Metricas modelo ganador

In [18]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [19]:
#Predicciones para Train y Test
y_train_pred = modelo_f.predict(X_train)
y_test_pred = modelo_f.predict(X_test)

In [20]:
# metricas conjunto Test
mae_test = mean_absolute_error(y_test, y_test_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
rmse_test = np.sqrt(mse_test)
r2_test = r2_score(y_test, y_test_pred)

In [21]:
#metricas de los dos conjuntos
df_metricas = pd.DataFrame({
    'Métrica': ['MAE (Error Absoluto Medio)', 'MSE (Error Cuadrático Medio)', 'RMSE (Raíz del MSE)', 'R² (Coef. Determinación)'],
    'Prueba (Test)': [f"{mae_test:.8f}", f"{mse_test:.8f}", f"{rmse_test:.8f}", f"{r2_test:.8f}"]
})

df_metricas

,Métrica,Prueba (Test)
0,MAE (Error Absoluto Medio),0.32617800
1,MSE (Error Cuadrático Medio),0.30520270
2,RMSE (Raíz del MSE),0.55245154
3,R² (Coef. Determinación),0.93945128




*   Mae = indica que el modelo se equivoco un 32% en sus prediciones.
*   MSE =  indica que existe poco margen de error con los datos no vistos, la desviación entre lo que predijo el modleo y los datos reales de prueba es reducida.
*   RMSE = indica el error promedio dando mas peso a los fallos grandes atipicos, en este caso la desviación es de solo 0.57 unidades
*  R²  =  indica que el modelo predice el 93.6% de los datos de manera correcta

De esta forma se puede concluir que el modelo ExtraTreesRegressor presenta un desempeño sobresaliente con el dataset que se obtuvo en la fase Load, permitiendo predecir la demanda atendida renovable en los 20 países con mayor consumo de América Latina utilizando la rejila temporal de 2000 a 2021.



# Guardar Modelo

In [22]:
# libreria para guardar el modelo
import joblib

In [23]:
ruta = "/content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/model/"
# Guardar modelo
joblib.dump(modelo_f,ruta + 'modelo_extratrees_energia.pkl')


['/content/drive/My Drive/Colab Notebooks/Master/1.ETL/entregable3/data/model/modelo_extratrees_energia.pkl']